# Exp2 IVMH vs EPOCH History Breakdown

This notebook reruns a deterministic history sweep for `IVMH` and representative `EPOCH-WR`, then plots stacked breakdowns of `InitLoad`, `Update`, `BuildSnap`, `HistoryScan`, and `RecentScan`.

Trace semantics per block:
- `RecentScan x 5`
- `MarkTs`
- `Update`

Sweep semantics:
- `HS = 0..10`
- There are 20 total blocks
- In every other block (2nd, 4th, 6th, ...), replace the first scan with `HistoryScan`
- Each `HistoryScan` reads the distinct readable timestamp published by the immediately preceding block


In [ ]:
from pathlib import Path
import sys
import importlib
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

ROOT = Path('../../').resolve()
sys.path.append(str(ROOT / 'benches'))
import sigmod_exp_common as _sigmod_exp_common
importlib.reload(_sigmod_exp_common)
sys.path.append(str(ROOT / 'benches' / 'hash_join' / 'htap_simulation'))

from sigmod_exp_common import (
    TOL,
    SIGMOD_BUCKET_NUM,
    apply_paper_style,
    current_run_stamp,
    ensure_dirs,
    run_checked,
)
from bench_script_functions import parse_result

apply_paper_style(ROOT)

EXP_DIR = (ROOT / 'benches' / 'sigmod_exp2_ivmh_history_breakdown').resolve()
DATA_DIR = EXP_DIR / 'data'
FIGS_DIR = EXP_DIR / 'figs'
ensure_dirs(DATA_DIR, FIGS_DIR)

BIN = ROOT / 'target' / 'release' / 'htap_ivmh_history_breakdown_wkld'

ALL_TABLE_SPECS = {
    'snap': {
        'table_type': 'naive',
        'repair_mode': 'no-repair',
        'label': 'SNAP',
    },
    'ivmh': {
        'table_type': 'ivmh',
        'repair_mode': 'no-repair',
        'label': 'IVMH',
    },
    'mono_wr': {
        'table_type': 'heap',
        'repair_mode': 'write-repair',
        'label': 'MONO-WR',
    },
    'dual_wr': {
        'table_type': 'chain',
        'repair_mode': 'write-repair',
        'label': 'DUAL-WR',
    },
    'epoch_wr': {
        'table_type': 'par',
        'repair_mode': 'write-repair',
        'label': 'EPOCH-WR',
    },
}

CONFIG = {
    'warehouse_count': 2,
    'bucket_num': SIGMOD_BUCKET_NUM,
    'update_ratio': 0.0001,
    'seed': 232323223,
    'blocks': 20,
    'scans_per_block': 5,
    'history_scans': list(range(0, 11, 2)),
    'repeat': 2,
    'warmup_runs': 1,
    'trim': 0,
    'timeout_sec': 900,
    'structures': ['snap', 'ivmh', 'mono_wr', 'dual_wr', 'epoch_wr'],
}

invalid_structures = [key for key in CONFIG['structures'] if key not in ALL_TABLE_SPECS]
if invalid_structures:
    raise ValueError(f'Unknown structure keys: {invalid_structures}')

TABLE_SPECS = [ALL_TABLE_SPECS[key] for key in CONFIG['structures']]

RUN_STAMP = current_run_stamp()
RUN_TAG = '_'.join([
    f"wc{CONFIG['warehouse_count']}",
    f"bn{CONFIG['bucket_num']}",
    f"ur{str(CONFIG['update_ratio']).replace('.', 'p')}",
    f"blk{CONFIG['blocks']}",
    f"spb{CONFIG['scans_per_block']}",
    f"rep{CONFIG['repeat']}",
    f"seed{CONFIG['seed']}",
    RUN_STAMP,
])

COLOR_MAP = {
    'InitLoad': TOL['grey'],
    'Update': TOL['yellow'],
    'BuildSnap': TOL['purple'],
    'HistoryScan': TOL['darkgreen'],
    'RecentScan': TOL['green'],
}
HATCH_MAP = {
    'HistoryScan': '\\\\',
    'RecentScan': '///',
}
STACK_ORDER = ['InitLoad', 'Update', 'BuildSnap', 'HistoryScan', 'RecentScan']
SERIES_STYLE = {
    'SNAP': (TOL['red'], ':', 'x'),
    'IVMH': (TOL['yellow'], '--', 'P'),
    'MONO-WR': (TOL['blue'], '-', 'o'),
    'DUAL-WR': (TOL['cyan'], '-', 's'),
    'EPOCH-WR': (TOL['green'], '-', 'D'),
}

print('ROOT       :', ROOT)
print('BIN        :', BIN)
print('OUTDIR     :', DATA_DIR)
print('CONFIG     :', CONFIG)
print('STRUCTURES :', [spec['label'] for spec in TABLE_SPECS])
print('STAMP      :', RUN_STAMP)
print('TAG        :', RUN_TAG)



In [ ]:
run_checked(
    ['cargo', 'build', '--release', '--bin', 'htap_ivmh_history_breakdown_wkld'],
    ROOT,
    timeout=CONFIG['timeout_sec'],
)
print('Built', BIN)


In [ ]:
def build_args(spec, history_scans: int):
    return [
        '--table-type', spec['table_type'],
        '--repair-mode', spec['repair_mode'],
        '--warehouse-count', str(CONFIG['warehouse_count']),
        '--update-ratio', str(CONFIG['update_ratio']),
        '--bucket-num', str(CONFIG['bucket_num']),
        '--seed', str(CONFIG['seed']),
        '--blocks', str(CONFIG['blocks']),
        '--scans-per-block', str(CONFIG['scans_per_block']),
        '--history-scans', str(history_scans),
    ]


def trim_trial_runs(df: pd.DataFrame) -> pd.DataFrame:
    trim = CONFIG['trim']
    if trim <= 0 or df.empty or 'trial' not in df.columns:
        return df
    totals = (
        df.groupby('trial', as_index=False)['duration_ms']
        .sum()
        .sort_values('duration_ms')
    )
    if len(totals) <= 2 * trim:
        return df
    keep = set(totals.iloc[trim:len(totals) - trim]['trial'])
    return df[df['trial'].isin(keep)].copy()


def classify_rows(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out['build_reason'] = out['build_reason'].fillna('')
    out['component'] = None
    out.loc[out['tx_type'] == 'InitLoad', 'component'] = 'InitLoad'
    out.loc[out['tx_type'] == 'Update', 'component'] = 'Update'
    out.loc[(out['tx_type'] == 'MarkTs') & (out['build_reason'] != ''), 'component'] = 'BuildSnap'
    out.loc[out['tx_type'] == 'HistoryScan', 'component'] = 'HistoryScan'
    out.loc[out['tx_type'] == 'RecentScan', 'component'] = 'RecentScan'
    return out


def aggregate_components(df: pd.DataFrame, label: str, history_scans: int) -> pd.DataFrame:
    tx_count = df['tx_id'].nunique()
    comp = (
        df[df['component'].notna()]
        .groupby('component', as_index=False)['duration_ms']
        .sum()
    )
    comp['table_label'] = label
    comp['history_scans'] = history_scans
    comp['duration_ms'] = comp['duration_ms'] / tx_count
    return comp[['table_label', 'history_scans', 'component', 'duration_ms']]


def run_case(spec, history_scans: int):
    args = [str(BIN), *build_args(spec, history_scans)]
    print(f"{spec['label']} history_scans = {history_scans}")
    print('cmd =', ' '.join(args))

    for warmup_idx in range(CONFIG['warmup_runs']):
        _ = run_checked(args, ROOT, timeout=CONFIG['timeout_sec'], quiet=True)
        print(f'  warmup {warmup_idx + 1}/{CONFIG["warmup_runs"]}')

    trials = []
    for trial in range(CONFIG['repeat']):
        result = run_checked(args, ROOT, timeout=CONFIG['timeout_sec'], quiet=True)
        df = parse_result(result.stdout, spec['table_type'])
        df['trial'] = trial
        df['table_label'] = spec['label']
        df['history_scans'] = history_scans
        trials.append(classify_rows(df))
        print(f'  trial {trial + 1}/{CONFIG["repeat"]}')

    raw = pd.concat(trials, ignore_index=True)
    trimmed = trim_trial_runs(raw)
    agg = aggregate_components(trimmed, spec['label'], history_scans)
    return raw, trimmed, agg


In [ ]:
raw_rows = []
trimmed_rows = []
stack_rows = []

for spec in TABLE_SPECS:
    for history_scans in CONFIG['history_scans']:
        raw_df, trimmed_df, agg_df = run_case(spec, history_scans)
        raw_rows.append(raw_df)
        trimmed_rows.append(trimmed_df)
        stack_rows.append(agg_df)

raw_df = pd.concat(raw_rows, ignore_index=True)
trimmed_df = pd.concat(trimmed_rows, ignore_index=True)
stack_df = pd.concat(stack_rows, ignore_index=True)

stamped_raw_csv = DATA_DIR / f'ivmh_epoch_history_breakdown_raw_{RUN_TAG}.csv'
latest_raw_csv = DATA_DIR / 'ivmh-epoch-history-breakdown-raw.csv'
stamped_stack_csv = DATA_DIR / f'ivmh_epoch_history_breakdown_{RUN_TAG}.csv'
latest_stack_csv = DATA_DIR / 'ivmh-epoch-history-breakdown.csv'

raw_df.to_csv(stamped_raw_csv, index=False)
raw_df.to_csv(latest_raw_csv, index=False)
stack_df.to_csv(stamped_stack_csv, index=False)
stack_df.to_csv(latest_stack_csv, index=False)

display(
    stack_df.pivot(index=['table_label', 'history_scans'], columns='component', values='duration_ms')
    .reindex(columns=STACK_ORDER[::-1], fill_value=0.0)
    .fillna(0.0)
)
print('Saved', stamped_raw_csv)
print('Saved', latest_raw_csv)
print('Saved', stamped_stack_csv)
print('Saved', latest_stack_csv)


In [ ]:
def build_component_legend_handles():
    handles = []
    labels = []
    for component in STACK_ORDER:
        if component in {'InitLoad', 'Update', 'BuildSnap'}:
            patch = Patch(facecolor=COLOR_MAP[component], edgecolor='black', linewidth=0.4)
        else:
            patch = Patch(facecolor='white', edgecolor=COLOR_MAP[component], linewidth=0.9, hatch=HATCH_MAP[component])
        handles.append(patch)
        labels.append(component)
    return handles, labels


def group_pivot(df, label):
    return (
        df[df['table_label'] == label]
        .pivot(index='history_scans', columns='component', values='duration_ms')
        .reindex(index=CONFIG['history_scans'], columns=STACK_ORDER, fill_value=0.0)
        .fillna(0.0)
    )


def build_series_legend_handles():
    handles = []
    labels = []
    for spec in TABLE_SPECS:
        label = spec['label']
        color, linestyle, marker = SERIES_STYLE[label]
        handles.append(
            Line2D([0], [0], color=color, linestyle=linestyle, marker=marker, linewidth=1.8, markersize=5.5)
        )
        labels.append(label)
    return handles, labels


component_handles, component_labels = build_component_legend_handles()
series_handles, series_labels = build_series_legend_handles()
pivots = {spec['label']: group_pivot(stack_df, spec['label']) for spec in TABLE_SPECS}
y_max_bar = max(float(pivot.sum(axis=1).max()) for pivot in pivots.values()) * 1.12

fig, (ax_bar, ax_line) = plt.subplots(
    2,
    1,
    figsize=(10.2, 6.8),
    gridspec_kw={'height_ratios': [3.1, 1.7]},
)
group_x = list(range(len(CONFIG['history_scans'])))
bar_width = 0.15
center = (len(TABLE_SPECS) - 1) / 2.0
offsets = {
    spec['label']: (idx - center) * (bar_width + 0.02)
    for idx, spec in enumerate(TABLE_SPECS)
}
totals = {}

for spec in TABLE_SPECS:
    label = spec['label']
    pivot = pivots[label]
    totals[label] = pivot.sum(axis=1).tolist()
    x = [v + offsets[label] for v in group_x]
    bottom = [0.0 for _ in x]
    for component in STACK_ORDER:
        values = pivot[component].tolist()
        if component in {'InitLoad', 'Update', 'BuildSnap'}:
            ax_bar.bar(x, values, bottom=bottom, width=bar_width, color=COLOR_MAP[component], edgecolor='black', linewidth=0.4)
        else:
            ax_bar.bar(x, values, bottom=bottom, width=bar_width, color='white', edgecolor='black', linewidth=0.4)
            ax_bar.bar(x, values, bottom=bottom, width=bar_width, color='none', edgecolor=COLOR_MAP[component], linewidth=0.9, hatch=HATCH_MAP[component])
        bottom = [b + v for b, v in zip(bottom, values)]
    for xi in x:
        ax_bar.text(xi, -0.085, label, ha='center', va='top', transform=ax_bar.get_xaxis_transform(), fontsize=6.6, rotation=90, clip_on=False)

ax_bar.set_xticks(group_x)
ax_bar.set_xticklabels([str(v) for v in CONFIG['history_scans']])
ax_bar.set_ylabel('Duration (ms / tx)')
ax_bar.set_xlabel('Historical Scans Inserted')
ax_bar.set_ylim(0, y_max_bar)
ax_bar.yaxis.grid(True, linestyle='--', linewidth=0.6, alpha=0.6)
component_legend = ax_bar.legend(component_handles, component_labels, loc='upper right', framealpha=0.95)
ax_bar.add_artist(component_legend)
ax_bar.legend(series_handles, series_labels, loc='upper left', framealpha=0.95, ncol=2)

for spec in TABLE_SPECS:
    label = spec['label']
    color, linestyle, marker = SERIES_STYLE[label]
    ax_line.plot(
        CONFIG['history_scans'],
        totals[label],
        label=label,
        color=color,
        linestyle=linestyle,
        marker=marker,
        linewidth=1.8,
        markersize=5.5,
    )

ax_line.set_xlabel('Historical Scans Inserted')
ax_line.set_ylabel('Total (ms / tx)')
ax_line.set_xticks(CONFIG['history_scans'])
ax_line.set_ylim(bottom=0)
ax_line.yaxis.grid(True, linestyle='--', linewidth=0.6, alpha=0.6)
ax_line.legend(loc='upper left', framealpha=0.95, ncol=3)
fig.tight_layout(rect=[0, 0.02, 1, 1])

stamped_pdf = FIGS_DIR / f'exp2_history_breakdown_compare_{RUN_TAG}.pdf'
latest_pdf = FIGS_DIR / 'exp2-history-breakdown-compare.pdf'
stamped_png = FIGS_DIR / f'exp2_history_breakdown_compare_{RUN_TAG}.png'
latest_png = FIGS_DIR / 'exp2-history-breakdown-compare.png'
fig.savefig(stamped_pdf, format='pdf', bbox_inches='tight')
fig.savefig(latest_pdf, format='pdf', bbox_inches='tight')
fig.savefig(stamped_png, dpi=220, bbox_inches='tight')
fig.savefig(latest_png, dpi=220, bbox_inches='tight')
plt.show()
print('Saved', stamped_pdf)
print('Saved', latest_pdf)
print('Saved', stamped_png)
print('Saved', latest_png)

